In [1]:
import pandas as pd
import numpy as np

## Dataset Loading

In [2]:
df = pd.read_csv("powerplant_data.csv")

In [3]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

## Data Splitting

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop("PE", axis=1)
y = df["PE"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Data Scaling

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
X_train_scaled

array([[ 0.74805289,  0.72006931, -0.32660017, -0.49711722],
       [ 0.86181948,  1.26515721, -0.98521113,  0.8181501 ],
       [ 0.93409473,  1.52314975,  0.32523844,  0.80167494],
       ...,
       [-0.22097078, -0.834965  ,  0.36756563, -0.83554456],
       [ 0.94747903,  1.14245344, -0.41971997, -0.45455637],
       [-1.77355014, -1.19049131,  1.92520594,  0.91837402]],
      shape=(7654, 4))

## Tensor Conversion

In [8]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

## Tensor Dataset Creation

In [9]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

## Tensor Data Load

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## Deep Learning : ANN Model

In [11]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            
            # 1st hidden layer :
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),
    
            # 2nd hidden layer :
            nn.Linear(6, 6),
            nn.ReLU(),
    
            # Output layer :
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)

## Loss & Optimizer

In [12]:
import torch.optim as optim 

model = ANN()

# Loss :
criterion = nn.MSELoss()

# Optimizer
optimizer = optim.Adam(model.parameters())

## Training the ANN

In [ ]:
train_losses = []
val_losses = []

epochs = 100

best_val_loss = float("inf")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation :

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs,yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if (epoch_val_loss < best_val_loss):
        best_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")

epoch 1/100 ==> train loss = 205399.81927083334 & val loss = 202152.28125
epoch 2/100 ==> train loss = 192163.5796875 & val loss = 177005.28125
epoch 3/100 ==> train loss = 154844.68743489584 & val loss = 132040.140625
epoch 4/100 ==> train loss = 114121.86682942709 & val loss = 101134.8359375
epoch 5/100 ==> train loss = 92891.05221354167 & val loss = 86105.625
epoch 6/100 ==> train loss = 76582.71077473958 & val loss = 65711.359375
epoch 7/100 ==> train loss = 51851.59424641927 & val loss = 35554.62109375
epoch 8/100 ==> train loss = 21186.756040445962 & val loss = 9352.076171875
epoch 9/100 ==> train loss = 4578.7925257364905 & val loss = 1995.979248046875
epoch 10/100 ==> train loss = 1394.3853996276855 & val loss = 1022.3267822265625
epoch 11/100 ==> train loss = 872.6136848449707 & val loss = 727.8052978515625
epoch 12/100 ==> train loss = 658.2833590189616 & val loss = 569.0875854492188
epoch 13/100 ==> train loss = 523.9311309814453 & val loss = 457.6988830566406
epoch 14/100 =

## Error - Graph :

In [ ]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "Training Loss": train_losses,
    "Validation Loss": val_losses
})

plt.plot(loss_df["Training Loss"], label = "Training Loss")
plt.plot(loss_df["Validation Loss"], label = "Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

## Load the best Model

In [ ]:
model.load_state_dict(torch.load("best_model.pt"))

## Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = criterion(train_preds, y_train_tensor)
    test_mse_loss = criterion(test_preds, y_test_tensor)

print("Training MSE : ", train_mse_loss.item())
print("Testing MSE : ", test_mse_loss.item())

In [ ]:
# R2 Score :

from sklearn.metrics import r2_score

print("r^2 score = ", r2_score(y_test, test_preds))

## Prediction Comparison

In [ ]:
predicted_df = pd.DataFrame(test_preds.numpy(), columns=["Predicted Values"])
actual_df = pd.DataFrame(y_test.values, columns=["Actual Values"])

pd.concat([predicted_df, actual_df], axis=1)